In [ ]:
import os

os.environ['KAGGLE_API_TOKEN'] = "Kaggle API"


In [ ]:
!kaggle competitions download -c super-ai-engineer-ss-6-heart-disease-prediction

!ls

!unzip super-ai-engineer-ss-6-heart-disease-prediction.zip -d super-ai-engineer-ss-6-heart-disease-prediction

!ls super-ai-engineer-ss-6-heart-disease-prediction

100% 4.30M/4.30M [00:00<00:00, 58.3MB/s]

sample_data  super-ai-engineer-ss-6-heart-disease-prediction.zip
Archive:  super-ai-engineer-ss-6-heart-disease-prediction.zip
  inflating: super-ai-engineer-ss-6-heart-disease-prediction/sample_submission.csv  
  inflating: super-ai-engineer-ss-6-heart-disease-prediction/test.csv  
  inflating: super-ai-engineer-ss-6-heart-disease-prediction/train.csv  
sample_submission.csv  test.csv  train.csv


In [ ]:
import pandas as pd
import os

for root, dirs, files in os.walk("super-ai-engineer-ss-6-heart-disease-prediction"):
    for file in files:
        print(os.path.join(root, file))

df = pd.read_csv('super-ai-engineer-ss-6-heart-disease-prediction/train.csv')
df.head()

super-ai-engineer-ss-6-heart-disease-prediction/sample_submission.csv
super-ai-engineer-ss-6-heart-disease-prediction/test.csv
super-ai-engineer-ss-6-heart-disease-prediction/train.csv


,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,train_000001,No,Yes,Yes,Yes,40.68,Yes,No,No,No,No,Yes,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes
1,train_000002,No,No,No,No,24.36,Yes,No,No,Yes,No,No,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No
2,train_000003,No,Yes,Yes,Yes,27.33,No,No,No,No,No,Yes,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes
3,train_000004,No,Yes,No,Yes,27.01,No,No,No,Yes,No,Yes,No,Good,No,Female,Some high school,"$35,000 to less than $50,000",74,Yes
4,train_000005,NaN,Yes,Yes,Yes,34.56,Yes,No,No,Yes,No,Yes,Yes,Very Poor,Yes,Male,Some high school,"$15,000 to less than $20,000",98,Yes


In [ ]:
!pip install pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pmdarima to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.1/486.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.8 M

In [ ]:
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('super-ai-engineer-ss-6-heart-disease-prediction/train.csv')
test_df = pd.read_csv('super-ai-engineer-ss-6-heart-disease-prediction/test.csv')
sample_sub = pd.read_csv('super-ai-engineer-ss-6-heart-disease-prediction/sample_submission.csv')
target_col = 'History of HeartDisease or Attack'

train_df = train_df.dropna(subset=[target_col]).reset_index(drop=True)
y = train_df[target_col].map({'Yes': 1, 'No': 0})
X = train_df.drop(columns=['ID', target_col])
X_test = test_df.drop(columns=['ID'])

def create_features(df):
    df_new = df.copy()
    risk_cols = ['High Blood Pressure', 'Told High Cholesterol', 'Smoked 100+ Cigarettes', 'Diagnosed Diabetes']

    risk_sum = np.zeros(len(df_new))
    for col in risk_cols:
        temp_val = df_new[col].map({'Yes': 1, 'No': 0}).fillna(0)
        risk_sum += temp_val

    df_new['Total_Risk_Score'] = risk_sum

    for col in df_new.columns:
        if df_new[col].dtype == 'object':
            df_new[col] = df_new[col].astype('category')
    return df_new

X = create_features(X)
X_test = create_features(X_test)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        random_state=42,
        n_estimators=400,
        learning_rate=0.03,
        class_weight='balanced',
        verbose=-1
    )
    model.fit(X_tr, y_tr)

    oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    print(f"Fold {fold+1}/5")

best_thresh = 0.5
best_f2 = 0
for t in np.arange(0.1, 0.6, 0.01):
    score = fbeta_score(y, (oof_preds >= t).astype(int), beta=2)
    if score > best_f2:
        best_f2 = score
        best_thresh = t

print(f"Golden Threshold Sum : {best_thresh:.2f} F2 Sum = {best_f2:.4f})")

final_predictions = np.where(test_preds >= best_thresh, 'Yes', 'No')
sample_sub[target_col] = final_predictions
sample_sub.to_csv('kfold_fe_heart_disease.csv', index=False)

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Golden Threshold Sum : 0.57 F2 Sum = 0.5428)
